# Week 4 Breakout Slides: Data Exploration

Before we can model data we need to *understand* it. This week we explore a dataset of country-level development indicators, focusing on two essential steps: measuring how variables **relate** to one another (correlation) and putting variables on a **common scale** (standardization). Both are groundwork for the unsupervised learning methods we'll build next.

## Import Libraries

As in Week 3, we lean on three core libraries: `numpy` for numerical arrays, `pandas` for tabular data, and `matplotlib` for plotting. pandas and matplotlib are both built on top of numpy, so the statistical tools carry over directly.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

## Load and Inspect the Data

We load a CSV of country development indicators into a pandas **DataFrame**. Each row is a country and each column is a metric — child mortality, exports, income, life expectancy, GDP per capita, and so on.

In [ ]:
path_to_file = Path('your/path')

In [ ]:
df = pd.read_csv(path_to_file / 'country_data.csv')

### Inspect the DataFrame

Displaying the DataFrame directly shows the first and last rows along with the column names and shape — a quick sanity check that the data loaded as expected.

In [ ]:
df

### Summary Statistics

`.describe()` runs count, mean, standard deviation, min, max, and the quartiles on every numeric column at once. Always look at this first — it reveals the scale of each variable and hints at skew or outliers.

In [ ]:
df.describe()

## Correlation

Correlation measures how strongly two variables move together, on a scale from **-1** (perfect inverse relationship) through **0** (no relationship) to **+1** (perfect positive relationship). pandas computes the full pairwise correlation matrix for every numeric column with a single `.corr()` call, and supports three different methods.

### Pearson Correlation

`.corr()` defaults to the **Pearson** coefficient, which measures the strength of a *linear* relationship. It's the right choice when variables are roughly normally distributed and related in a straight-line fashion.

In [ ]:
df.corr()

### Spearman Correlation

**Spearman** correlation ranks the values first, then correlates the ranks. This captures any *monotonic* relationship, one variable consistently increasing with the other, even when that relationship is curved rather than straight. It's also more robust to outliers.

In [ ]:
df.corr(method='spearman')

### Kendall Correlation

**Kendall's tau** is another rank-based measure, built on counting *concordant* and *discordant* pairs of observations. It tends to be more conservative than Spearman and is well suited to small samples.

In [ ]:
df.corr(method='kendall')

### Numpy Backup

If you're working with raw arrays instead of a DataFrame, NumPy's `np.corrcoef` computes the same Pearson coefficients. It expects each *variable* in a row, so we transpose (`.T`) the two columns we pull out before passing them in.

In [ ]:
np.corrcoef(df.loc[:, ['child_mort', 'exports']].T.values)

## Visualizing the Correlation Matrix

A grid of numbers is hard to scan for patterns. A **heatmap** encodes each correlation as a color, letting strong positive and negative relationships jump out at a glance. We build one up from a rough first pass to a polished, publication-ready figure.

### A First Attempt

`plt.pcolor` gives us a colored grid in one line — but with no labels, no colorbar, and a default color scale, it's hard to interpret. A useful starting point, not a finished product.

In [ ]:
plt.pcolor(df.corr())

### A Better Heatmap

With a bit more Matplotlib we get a readable figure: `imshow` draws the grid, a diverging `coolwarm` colormap fixed to the -1…1 range makes the sign of each correlation obvious, a colorbar provides reference, and the axes are labeled with the feature names.

In [ ]:
df_corr = df.corr()

fig, ax = plt.subplots(figsize=(6, 5))

# Use imshow to create the heatmap grid (-1 to 1 bounds for correlation)
im = ax.imshow(df_corr, cmap='coolwarm', vmin=-1, vmax=1)

# Add colorbar for reference
cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.set_ylabel("Pearson's R", rotation=-90, va="bottom")

# Set tick labels to show the feature names
ax.set_xticks(np.arange(len(df_corr.columns)))
ax.set_yticks(np.arange(len(df_corr.columns)))
ax.set_xticklabels(df_corr.columns)
ax.set_yticklabels(df_corr.columns)

# Rotate the tick labels on the x-axis for readability
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

ax.set_title("Correlation Matrix Heatmap")
fig.tight_layout()
plt.show()

### Best: Mask Redundant Values

A correlation matrix is symmetric and its diagonal is always 1, so half the grid is redundant. We build a boolean mask of the upper triangle with `np.triu_indices_from`, set those entries to `NaN` so they render blank, and are left with a clean lower-triangle heatmap showing each pair exactly once.

In [ ]:
df_corr = df.corr()

mask = np.zeros_like(df_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
df_corr[mask] = np.nan

fig, ax = plt.subplots(figsize=(6, 5))

# Use imshow to create the heatmap grid (-1 to 1 bounds for correlation)
im = ax.imshow(df_corr, cmap='coolwarm', vmin=-1, vmax=1)

# Add colorbar for reference
cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.set_ylabel("Pearson's R", rotation=-90, va="bottom")

# Set tick labels to show the feature names
ax.set_xticks(np.arange(len(df_corr.columns)))
ax.set_yticks(np.arange(len(df_corr.columns)))
ax.set_xticklabels(df_corr.columns)
ax.set_yticklabels(df_corr.columns)

# Rotate the tick labels on the x-axis for readability
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

ax.set_title("Correlation Matrix Heatmap")
fig.tight_layout()
plt.show()

## Distributions and Scaling

Correlation tells us how variables relate; next we look at how each one is *distributed*. This matters because many modeling techniques — especially the distance-based clustering methods coming next week — are thrown off when features live on wildly different scales.

### Boxplots on the Raw Data

Plotting boxplots of the raw columns side by side is nearly useless here: income and GDP per capita run into the tens of thousands while rates like inflation and fertility sit near single digits. The large-magnitude features flatten everything else against the axis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

df.boxplot()

### Standardization (Z-scores)

**Standardization** rescales each column to a *z-score*: subtract the column mean and divide by its standard deviation. Every feature ends up centered at 0 with a standard deviation of 1, so they can be compared — and clustered — on equal footing regardless of their original units.

In [ ]:
df_z = (df - df.mean()) / df.std()

### Boxplots on the Standardized Data

On the standardized data the boxplots are finally comparable. We can now read the *shape* of each distribution side by side, though note we've traded away the original magnitudes to get there.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

df_z.plot.box(showfliers=False, whis=[10, 90], ax=ax)

ax.set_ylim([-3, 3])